In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader

In [ ]:
def load_pdf(data):
    loader = PyPDFLoader(data)
    documents = loader.load()
    return documents    

In [ ]:
docs = load_pdf("../data/Medical_book.pdf")
docs[-1]

In [ ]:
from langchain.schema import Document
from typing import List

In [ ]:
def filter_documents(documents: List[Document]) -> List[Document]:
    """ Given a list of documents objects, return a list of documents 
    with only the page content and source metadata. """
    filtered_docs = []
    for doc in documents:
        src = doc.metadata.get("source", "")
        filtered_docs.append(
                             Document(
                                 page_content=doc.page_content,
                                 metadata={"source": src}
                             ))
    return filtered_docs

In [ ]:
minimal_docs = filter_documents(docs)

#### create text chunks

In [ ]:
def text_splitter(documents: List[Document]) -> List[Document]:
    """ Given a list of documents objects, return a list of documents with only the page content and source metadata. """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunk = text_splitter.split_documents(documents)
    return text_chunk

In [ ]:
text_chunks = text_splitter(minimal_docs)   
print("length of text chunks: ", len(text_chunks))

#### load embeddings model

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def get_embeddings_vector():
    """Load and return a HuggingFace embedding model."""
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings_model = get_embeddings_vector()

#### load env variables

In [ ]:
### load environments variables
import os

from dotenv import load_dotenv
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

#### create pinecone object

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pinecone = Pinecone(api_key=pinecone_api_key)


#### create index

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-bot"
serverless_spec = ServerlessSpec(cloud="aws", region="us-east-1")
if not pinecone.has_index(index_name):
    pinecone.create_index(index_name, 
                          dimension=384, 
                          metric="cosine",
                          spec=serverless_spec)
index = pinecone.Index(index_name)

#### upsert vectors of pinecone

In [ ]:
from langchain_pinecone import PineconeVectorStore
def create_vectorstore(index_name, model, text_chunks):
    vectorstore = PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=model,
        index_name=index_name
    )
    return vectorstore

vectorstore = create_vectorstore(index_name, embeddings_model, text_chunks)

#### retrieveal from documents/vector

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

In [ ]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [42]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a chemical from the pituitary gland, leading to increased growth in bone and soft tissue. It occurs when this abnormality happens after bone growth stops, typically in adults. Gigantism, on the other hand, occurs when this abnormality happens before bone growth stops, typically in children, resulting in unusual height.
